In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.stats import zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from umap import UMAP
import seaborn as sns
import matplotlib.pyplot as plt

## INSPIRE Path Definitions

In [2]:
# All the input paths
inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
ops_path = inspire_path / "operations.csv"
labs_path = inspire_path / "labs.csv"
vitals_path = inspire_path / "vitals.csv"
ward_vitals_path = inspire_path / "ward_vitals.csv"
combined_path = inspire_path / "combined_data.csv"
combined_cleaned_path = inspire_path / "combined_cleaned_data.csv"
pca_path = inspire_path / "pca_data.csv"
diagnosis_path = inspire_path / "diagnosis.csv"

# Output path
output_path = Path("/home/server/Projects/data/AKI")

In [3]:
# Read the necessary csv files for the preop data
df_lab = pd.read_csv(labs_path.as_posix())
df_ward_vital = pd.read_csv(ward_vitals_path.as_posix())
df_ops = pd.read_csv(ops_path.as_posix())
df_diags = pd.read_csv(diagnosis_path.as_posix())

In [4]:
# Keeping only the necessary columns
desired_columns = [
    'op_id', 'subject_id', 'age', 'sex', 'height', 'weight', 
    'asa', 'emop', 'opstart_time', 'opend_time', 
    'inhosp_death_time', 'allcause_death_time', 'orin_time', 'orout_time',
]

# Select only the desired columns
df_ops = df_ops[desired_columns]

In [5]:
# Add BSA (Body Surface Area) and BMI (Body Mass Index) columns

# Ensure height and weight are valid (not NaN) for calculations
valid_mask = df_ops['height'].notna() & df_ops['weight'].notna()

# Initialize BSA and BMI with NaN
df_ops['BSA'] = np.nan
df_ops['BMI'] = np.nan

# Calculate BSA and BMI only for valid rows
df_ops.loc[valid_mask, 'BSA'] = np.sqrt((df_ops.loc[valid_mask, 'height'] * df_ops.loc[valid_mask, 'weight']) / 3600)
df_ops.loc[valid_mask, 'BMI'] = df_ops.loc[valid_mask, 'weight'] / ((df_ops.loc[valid_mask, 'height'] / 100) ** 2)

In [6]:
# Add Booking Case Length column

valid_mask = df_ops['orin_time'].notna() & df_ops['orout_time'].notna()

df_ops['booking_case_length'] = np.nan
df_ops.loc[valid_mask, 'booking_case_length'] = df_ops.loc[valid_mask, 'orout_time'] - df_ops.loc[valid_mask, 'orin_time']

# Remove orin_time and orout_time columns
df_ops = df_ops.drop(columns=['orin_time', 'orout_time'])


In [7]:
# Filter cardiovascular diagnoses (ICD-10-CM codes starting with 'I')
df_diags_cvd = df_diags[df_diags['icd10_cm'].str.startswith('I', na=False)]

# Merge operations and cardiovascular diagnoses on subject_id
merged = pd.merge(
    df_ops[['op_id', 'subject_id', 'opstart_time']],
    df_diags_cvd[['subject_id', 'chart_time']],
    on='subject_id',
    how='inner'
)

# Filter diagnoses where chart_time < opstart_time
merged = merged[merged['chart_time'] < merged['opstart_time']]

# Count the number of diagnoses for each operation
num_card_events = merged.groupby('op_id').size().reset_index(name='num_card_events')

# Merge the counts back into the operations DataFrame
df_ops = pd.merge(
    df_ops,
    num_card_events,
    on='op_id',
    how='left'
)

# Fill NaN values with 0 for operations with no past cardiovascular diagnoses
df_ops['num_card_events'] = df_ops['num_card_events'].fillna(0).astype(int)

In [8]:
# TODO confirm that this works

# Define 90 days in minutes
ninety_days_in_minutes = 90 * 24 * 60

# Filter for rows where item_name is 'creatinine'
df_lab_creatinine = df_lab[df_lab['item_name'] == 'creatinine']

# Sort df_lab_creatinine by subject_id and chart_time
df_lab_creatinine = df_lab_creatinine.sort_values(by=['subject_id', 'chart_time'])

# Merge operations and labs data on subject_id
merged = pd.merge(
    df_ops[['op_id', 'subject_id', 'opstart_time']],
    df_lab_creatinine[['subject_id', 'chart_time', 'value']],
    on='subject_id',
    how='inner'
)

# Filter labs within the 90-day window
merged = merged[
    (merged['chart_time'] < merged['opstart_time']) &
    (merged['chart_time'] >= merged['opstart_time'] - ninety_days_in_minutes)
]

# Find the closest chart_time for each operation
closest_labs = (
    merged.loc[merged.groupby('op_id')['chart_time'].idxmax()]
    [['op_id', 'value']]
    .rename(columns={'value': 'last_preop_scr'})
)

# Merge the closest labs back into the operations DataFrame
df_ops = pd.merge(
    df_ops,
    closest_labs,
    on='op_id',
    how='left'
)


In [9]:
# Define 90 days in minutes
ninety_days_in_minutes = 90 * 24 * 60

# Filter for rows where item_name is 'creatinine'
df_lab_creatinine = df_lab[df_lab['item_name'] == 'creatinine']

# Sort df_lab_creatinine by subject_id and chart_time
df_lab_creatinine = df_lab_creatinine.sort_values(by=['subject_id', 'chart_time'])

# Merge operations and labs data on subject_id
merged = pd.merge(
    df_ops[['op_id', 'subject_id', 'opstart_time']],
    df_lab_creatinine[['subject_id', 'chart_time', 'value']],
    on='subject_id',
    how='inner'
)

# Filter labs within the 90-day window
merged = merged[
    (merged['chart_time'] < merged['opstart_time']) &
    (merged['chart_time'] >= merged['opstart_time'] - ninety_days_in_minutes)
]

# Find the minimum sCr value within the 90-day window for each operation
min_labs = (
    merged.groupby('op_id', as_index=False)
    .agg({'value': 'min'})
    .rename(columns={'value': 'min_preop_scr'})
)

# Merge the minimum labs back into the operations DataFrame
df_ops = pd.merge(
    df_ops,
    min_labs,
    on='op_id',
    how='left'
)



In [10]:
# Output currrent results

df_ops.to_csv(output_path / "preop_data.csv", index=False)

## Time series data parsing

In [ ]:
# Load in vitals
df_vitals = pd.read_csv(vitals_path.as_posix())

# Output path
output_path_intraop = Path("/home/server/Projects/data/AKI/intraop_data")

In [ ]:
# Get the unique item names
unique_items = df_vitals['item_name'].unique()
# Iterate over each unique item
for item in unique_items:
    # Filter the data for the current item
    item_data = df_vitals[df_vitals['item_name'] == item]
    
    # Group by op_id and create time series
    grouped = item_data.groupby('op_id')
    
    for op_id, group in grouped:
        # Create a directory for the current op_id
        op_dir = output_path_intraop / f"op_{op_id}"
        op_dir.mkdir(parents=True, exist_ok=True)
        
        # Sort by chart_time to ensure time series is ordered
        group = group.sort_values(by='chart_time')
        
        # Define the output file path for the current item
        output_file = op_dir / f"{item}.csv"
        
        # Save the time series to the file
        group[['chart_time', 'value']].to_csv(output_file, index=False)

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool

# Define paths
inspire_path = Path("/home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3")
output_path = Path("/home/server/Projects/data/AKI")
vitals_path = inspire_path / "vitals.csv"

# Output file for combined data
output_file = output_path / "combined_vitals.parquet"

# Define a function to process each group
def process_group(group_data):
    """
    Sorts the group by 'chart_time' and returns the processed DataFrame.
    
    Args:
        group_data: Tuple containing (group_key, group_df).
    
    Returns:
        pd.DataFrame: Processed group data.
    """
    (op_id, item_name), group = group_data
    group = group.sort_values(by='chart_time')  # Sort by chart_time
    return group[['op_id', 'item_name', 'chart_time', 'value']]

# Load in vitals
print(f"Loading vitals from {vitals_path}")
df_vitals = pd.read_csv(vitals_path.as_posix())

# Group by both 'op_id' and 'item_name'
print("Grouping data...")
grouped = list(df_vitals.groupby(['op_id', 'item_name']))

# Preallocate results array
results = [None] * len(grouped)

# Wrapper function to fill results array by index
def process_and_store(args):
    idx, group_data = args
    results[idx] = process_group(group_data)

# Parallel processing
print("Processing groups in parallel...")
with Pool() as pool:
    list(
        tqdm(
            pool.imap_unordered(process_and_store, enumerate(grouped)),
            total=len(grouped),
            desc="Processing Groups"
        )
    )

Loading vitals from /home/server/Projects/data/INSPIRE/physionet.org/files/inspire/1.3/vitals.csv
Grouping data...
Processing groups in parallel...


Processing Groups:  18%|█▊        | 509995/2881970 [03:01<23:18:56, 28.26it/s]

In [ ]:
# Combine all results into a single DataFrame
print("Combining results...")
structured_df = pd.concat(results, ignore_index=True)

# Save the combined data as a Parquet file
structured_df.to_parquet(output_file, index=False)

print(f"Combined data saved to {output_file}")

## Tests to confirm data is cleaned correctly

In [10]:
# Checking that number of cardio events accumulate correctly with time

not_zero_rows = df_ops[df_ops['num_card_events'] != 0]
not_zero_rows = not_zero_rows.sort_values(by=['subject_id', 'opstart_time'])

# Identify subject_ids where num_card_events changes across rows
subject_ids_with_changes = not_zero_rows.groupby('subject_id')['num_card_events'].nunique()
subject_ids_with_changes = subject_ids_with_changes[subject_ids_with_changes > 1].index

# Filter rows where subject_id is in the identified list
rows_with_changes = not_zero_rows[not_zero_rows['subject_id'].isin(subject_ids_with_changes)]

# Display the result)
rows_with_changes.head(30)

,op_id,subject_id,age,sex,height,weight,asa,emop,opstart_time,opend_time,inhosp_death_time,allcause_death_time,BSA,BMI,booking_case_length,num_card_events,last_preop_scr
3920,469691013,100016333,75,M,170.0,70.0,2.0,0,1950.0,2040.0,NaN,5610240.0,1.818119,24.221453,135.0,6,0.77
11797,478614166,100016333,75,M,170.0,70.0,2.0,0,330935.0,331030.0,NaN,5610240.0,1.818119,24.221453,135.0,17,1.05
17379,442943155,100016333,75,M,170.0,70.0,2.0,0,542255.0,542675.0,NaN,5610240.0,1.818119,24.221453,480.0,27,0.98
87775,496524177,100025752,50,M,175.0,80.0,3.0,0,20745.0,21235.0,NaN,2046240.0,1.972027,26.122449,595.0,13,5.55
98079,477744827,100025752,50,M,175.0,80.0,3.0,1,441875.0,441960.0,NaN,2046240.0,1.972027,26.122449,105.0,31,5.55
99322,462016574,100025752,50,M,175.0,80.0,NaN,1,494420.0,494515.0,NaN,2046240.0,1.972027,26.122449,135.0,33,3.04
34545,446600407,100106310,75,M,165.0,60.0,3.0,0,9705.0,10130.0,NaN,NaN,1.658312,22.038567,525.0,5,0.88
36682,473184017,100106310,75,M,165.0,65.0,2.0,0,91460.0,91500.0,NaN,NaN,1.726026,23.875115,70.0,7,0.80
83882,496605180,100109671,75,M,170.0,75.0,4.0,1,1230.0,1455.0,NaN,NaN,1.881932,25.951557,300.0,2,0.84
95728,406365862,100109671,75,M,170.0,75.0,3.0,0,471760.0,471815.0,NaN,NaN,1.881932,25.951557,95.0,18,0.92


In [ ]:
# Checking to see the correspodence between inhosp_death_time and allcause_death_time

# Filter rows where inhosp_death_time is not NaN
non_nan_rows = df_ops[df_ops['inhosp_death_time'].notna()]

# Display the first 10 rows of the filtered DataFrame
non_nan_rows.head(10)

,op_id,subject_id,age,sex,height,weight,asa,emop,opstart_time,opend_time,inhosp_death_time,allcause_death_time,BSA,BMI,booking_case_length
1,446270725,158995752,70,M,170.0,45.0,NaN,1,1370.0,1540.0,69860.0,106560.0,1.457738,15.570934,210.0
35,452566478,158995752,70,M,175.0,45.0,3.0,1,3835.0,4015.0,69860.0,106560.0,1.479020,14.693878,200.0
121,469321477,127354791,70,M,175.0,85.0,1.0,0,4870.0,5390.0,1326345.0,1363680.0,2.032718,27.755102,605.0
175,435330158,195522551,50,F,165.0,80.0,1.0,0,1955.0,1975.0,268065.0,282240.0,1.914854,29.384757,60.0
223,494888235,124379720,65,M,170.0,45.0,NaN,1,44695.0,44740.0,67495.0,112320.0,1.457738,15.570934,90.0
237,422875661,165887534,60,M,165.0,65.0,NaN,1,30070.0,30130.0,215410.0,246240.0,1.726026,23.875115,90.0
255,428885515,158995752,70,M,175.0,45.0,NaN,0,10960.0,11030.0,69860.0,106560.0,1.479020,14.693878,110.0
326,493495342,185361530,60,M,160.0,45.0,NaN,0,3780.0,4220.0,25255.0,93600.0,1.414214,17.578125,510.0
356,425582509,185903271,50,F,155.0,50.0,2.0,1,11050.0,11180.0,325440.0,362880.0,1.467235,20.811655,195.0
385,496636079,174773781,40,F,170.0,50.0,2.0,0,3485.0,3550.0,325805.0,325440.0,1.536591,17.301038,90.0


In [20]:
df_ops.columns

Index(['op_id', 'subject_id', 'age', 'sex', 'height', 'weight', 'asa', 'emop',
       'opstart_time', 'opend_time', 'inhosp_death_time',
       'allcause_death_time', 'BSA', 'BMI', 'booking_case_length'],
      dtype='object')